# 🔄 All Encoders & Scalers — Comparison Cheat Sheet
---
> **Dataset:** `tips` (seaborn) + `titanic`

## 📌 Quick Decision Guide

### Which Encoder?
| Situation | Use |
|---|---|
| Target column (y) | `LabelEncoder` |
| Multiple cols, ordinal data (Low/Med/High) | `OrdinalEncoder` |
| Nominal data, sklearn pipeline | `OneHotEncoder` |
| Quick EDA / not in pipeline | `pd.get_dummies` |

### Which Scaler?
| Situation | Use |
|---|---|
| Default choice (most ML models) | `StandardScaler` |
| Need values in [0,1] (neural nets) | `MinMaxScaler` |
| Data has significant outliers | `RobustScaler` |
| Tree-based models (RF, XGB, DT) | No scaling needed ✅ |

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import (LabelEncoder, OrdinalEncoder, OneHotEncoder,
                                    StandardScaler, MinMaxScaler, RobustScaler)

df = sns.load_dataset('tips')
print("Original dataset:")
print(df.head(3))
print("\nCategorical cols:", df.select_dtypes('object').columns.tolist())
print("Numeric cols    :", df.select_dtypes('number').columns.tolist())

In [ ]:
# ── LabelEncoder ──
df1 = sns.load_dataset('tips')
lb = LabelEncoder()
for col in ['sex','smoker','day','time']:
    df1[col] = lb.fit_transform(df1[col])
print("1. LabelEncoder output:")
print(df1.head(3), "\n")

In [ ]:
# ── OrdinalEncoder ──
df2 = sns.load_dataset('tips')[['sex','smoker','day','time']]
oe = OrdinalEncoder(categories=[['Male','Female'],['No','Yes'],
                                 ['Sat','Sun','Thur','Fri'],['Dinner','Lunch']])
df2_enc = pd.DataFrame(oe.fit_transform(df2), columns=df2.columns)
print("2. OrdinalEncoder output:")
print(df2_enc.head(3), "\n")

In [ ]:
# ── OneHotEncoder ──
df3 = sns.load_dataset('tips')
ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int32)
enc = ohe.fit_transform(df3[['sex','smoker','day','time']])
cols = ohe.get_feature_names_out(['sex','smoker','day','time'])
df3_enc = pd.DataFrame(enc, columns=cols)
print("3. OneHotEncoder columns:", list(cols))
print(df3_enc.head(3), "\n")

In [ ]:
# ── get_dummies ──
df4 = sns.load_dataset('tips')
df4 = pd.get_dummies(df4, columns=['sex','smoker','day','time'], drop_first=True).astype(int)
print("4. get_dummies columns:", list(df4.columns))
print(df4.head(3), "\n")

In [ ]:
# ── Scalers visual comparison ──
df5 = sns.load_dataset('tips')
col = df5['total_bill'].values.reshape(-1, 1)

ss = StandardScaler().fit_transform(col)
mn = MinMaxScaler().fit_transform(col)
rb = RobustScaler().fit_transform(col)

fig, axes = plt.subplots(1, 4, figsize=(16,4))
for ax, data, title, color in zip(axes,
    [col, ss, mn, rb],
    ['Original', 'StandardScaler\n(mean=0, std=1)', 'MinMaxScaler\n[0,1]', 'RobustScaler\n(median/IQR)'],
    ['steelblue','green','orange','red']):
    ax.hist(data, bins=20, color=color, alpha=0.8)
    ax.set_title(title, fontsize=10)
plt.suptitle('total_bill Distribution After Each Scaler', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Titanic dataset: Real-world encoding example ──
df_t = pd.read_csv("/content/titanic - titanic.csv")
print("Titanic shape:", df_t.shape)
print("\nNull values:")
print(df_t.isnull().sum()[df_t.isnull().sum() > 0])

In [ ]:
# Titanic: handle nulls + encode + scale
df_t = pd.read_csv("/content/titanic - titanic.csv")

# Drop high-null & irrelevant cols
df_t = df_t.drop(columns=['Cabin','Name','Ticket','PassengerId'], errors='ignore')

# Fill missing values
df_t['Age'].fillna(df_t['Age'].median(), inplace=True)
df_t['Embarked'].fillna(df_t['Embarked'].mode()[0], inplace=True)

# LabelEncoder on binary cols
lb = LabelEncoder()
df_t['Sex'] = lb.fit_transform(df_t['Sex'])

# get_dummies on nominal col
df_t = pd.get_dummies(df_t, columns=['Embarked'], drop_first=True).astype({
    col: int for col in df_t.columns if df_t[col].dtype == bool})

print("Titanic after encoding:")
print(df_t.head(3))
print("\nShape:", df_t.shape)

## 📋 Final Summary
| Encoder | Input | Output | Key Param |
|---|---|---|---|
| `LabelEncoder` | 1 col string | integers | — |
| `OrdinalEncoder` | N cols strings | integers with order | `categories` |
| `OneHotEncoder` | N cols strings | binary cols | `drop='first'` |
| `pd.get_dummies` | DataFrame | binary cols | `drop_first=True` |

| Scaler | Formula | Range | Best For |
|---|---|---|---|
| `StandardScaler` | (x-mean)/std | unbounded | Linear/SVM/KNN |
| `MinMaxScaler` | (x-min)/(max-min) | [0,1] | Neural nets |
| `RobustScaler` | (x-median)/IQR | unbounded | Outlier data |